# COVID-19 Global Impact Dashboard

**Veda Technology — Major Project**

This notebook implements the complete analytics pipeline requested in the project brief:

- public COVID-19 data from **Our World in Data (OWID)**
- cleaning and missing-value handling
- population-normalized comparisons
- 7-day rolling averages
- case fatality rate (CFR)
- doubling-time calculation
- vaccination rollout analysis
- wave/peak identification
- country and period comparison
- export of cleaned data and charts

> **Reproducibility note:** the notebook downloads the current OWID compact COVID-19 dataset at runtime. The included static preview files are only for immediate portfolio presentation; the notebook remains the source of truth for the full time-series analysis.


## 1. Business / analytical question

**Primary question:** How did COVID-19 waves and vaccination rollout differ across countries, and what patterns can be observed when cases and deaths are normalized by population?

A focused secondary question is:

> **Did higher vaccination coverage coexist with lower cumulative confirmed-death rates?**

This is treated as an **exploratory association**, not a causal claim. Differences in reporting, testing, age structure, healthcare capacity, variants, prior infection, and timing can affect the observed relationship.


## 2. Data source

Our World in Data states that its COVID-19 data are consolidated from multiple sources. Current cases and deaths are based on WHO data, while vaccination indicators are compiled from official sources and WHO.

OWID's current technical documentation provides a compact COVID-19 dataset and separate cases/deaths and vaccination datasets.

**Primary dataset:**  
`https://catalog.ourworldindata.org/garden/covid/latest/compact/compact.csv`

**Documentation:**  
`https://docs.owid.io/projects/etl/api/covid/`


In [ ]:
# 3. Imports and configuration
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

OWID_URL = "https://catalog.ourworldindata.org/garden/covid/latest/compact/compact.csv"

# Countries used for the main dashboard examples.
FOCUS_COUNTRIES = [
    "India", "United States", "Brazil", "United Kingdom",
    "Germany", "France", "Italy", "Japan", "South Korea", "Australia"
]

print("Configuration ready.")


In [ ]:
# 4. Load the OWID compact dataset
df_raw = pd.read_csv(OWID_URL)

print("Raw shape:", df_raw.shape)
display(df_raw.head())
print("\nColumns:")
print(df_raw.columns.tolist())


## 5. Cleaning strategy

The cleaning rules are deliberately conservative:

1. Standardize the date field.
2. Sort by country and date.
3. Convert numeric indicators with `to_numeric(errors="coerce")`.
4. Remove duplicate country-date records.
5. Do **not** replace true missing observations with zero.
6. Use zero only where a metric is logically zero and the source explicitly represents it that way.
7. For derived rates, divide only when the denominator is positive.
8. Keep a clear distinction between missing data and zero values.


In [ ]:
# 5.1 Basic cleaning
df = df_raw.copy()

# Normalize common column names if present.
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Keep only rows with a usable location/date where those fields exist.
required = [c for c in ["location", "date"] if c in df.columns]
df = df.dropna(subset=required)

# Numeric conversion for all non-key columns.
key_cols = [c for c in ["location", "date", "iso_code"] if c in df.columns]
for col in df.columns:
    if col not in key_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove duplicate location-date observations.
dedup_cols = [c for c in ["location", "date"] if c in df.columns]
df = df.sort_values(dedup_cols).drop_duplicates(dedup_cols, keep="last")

print("Cleaned shape:", df.shape)
print("Missing-value rate (top 20 columns):")
display(df.isna().mean().sort_values(ascending=False).head(20))


In [ ]:
# 6. Detect useful columns across OWID schema versions
def first_existing(candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

COL = {
    "location": first_existing(["location"]),
    "date": first_existing(["date"]),
    "population": first_existing(["population", "total_population"]),
    "new_cases": first_existing(["new_cases", "weekly_cases"]),
    "total_cases": first_existing(["total_cases"]),
    "new_deaths": first_existing(["new_deaths", "weekly_deaths"]),
    "total_deaths": first_existing(["total_deaths"]),
    "people_vaccinated": first_existing(["people_vaccinated", "total_people_vaccinated"]),
    "people_fully_vaccinated": first_existing([
        "people_fully_vaccinated", "total_people_fully_vaccinated"
    ]),
}

display(pd.DataFrame({"logical_metric": list(COL.keys()), "source_column": list(COL.values())}))


In [ ]:
# 7. Select the focus countries and create derived metrics
work = df[df["location"].isin(FOCUS_COUNTRIES)].copy()
work = work.sort_values(["location", "date"])

# 7-day rolling averages
if COL["new_cases"]:
    work["new_cases_7d"] = (
        work.groupby("location")[COL["new_cases"]]
        .transform(lambda s: s.rolling(7, min_periods=4).mean())
    )

if COL["new_deaths"]:
    work["new_deaths_7d"] = (
        work.groupby("location")[COL["new_deaths"]]
        .transform(lambda s: s.rolling(7, min_periods=4).mean())
    )

# Population-normalized daily metrics
if COL["new_cases"] and COL["population"]:
    work["new_cases_per_million"] = work[COL["new_cases"]] / work[COL["population"]] * 1_000_000

if COL["new_deaths"] and COL["population"]:
    work["new_deaths_per_million"] = work[COL["new_deaths"]] / work[COL["population"]] * 1_000_000

# Cumulative per-million metrics
if COL["total_cases"] and COL["population"]:
    work["total_cases_per_million"] = work[COL["total_cases"]] / work[COL["population"]] * 1_000_000

if COL["total_deaths"] and COL["population"]:
    work["total_deaths_per_million"] = work[COL["total_deaths"]] / work[COL["population"]] * 1_000_000

# Crude case fatality rate
if COL["total_cases"] and COL["total_deaths"]:
    work["case_fatality_rate_pct"] = np.where(
        work[COL["total_cases"]] > 0,
        work[COL["total_deaths"]] / work[COL["total_cases"]] * 100,
        np.nan
    )

# Doubling time:  ln(2) / ln(current cumulative cases / previous cumulative cases)
# Calculated over a 7-day interval to reduce daily reporting noise.
if COL["total_cases"]:
    prev_cases = work.groupby("location")[COL["total_cases"]].shift(7)
    growth_factor = work[COL["total_cases"]] / prev_cases
    work["doubling_time_days"] = np.where(
        (growth_factor > 1) & np.isfinite(growth_factor),
        7 * np.log(2) / np.log(growth_factor),
        np.nan
    )

print("Derived metrics created.")
display(work.head())


In [ ]:
# 8. Example: cases and deaths with 7-day smoothing
if "new_cases_7d" in work.columns and "new_deaths_7d" in work.columns:
    country = "India"
    plot_df = work[work["location"] == country].copy()

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(plot_df["date"], plot_df["new_cases_7d"], label="Cases — 7-day average")
    ax.set_title(f"{country}: 7-Day Rolling Average of New Confirmed Cases")
    ax.set_xlabel("Date")
    ax.set_ylabel("New cases")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    plt.show()


In [ ]:
# 9. Compare countries on population-normalized cumulative deaths
if "total_deaths_per_million" in work.columns:
    latest = (
        work.dropna(subset=["total_deaths_per_million"])
        .sort_values("date")
        .groupby("location", as_index=False)
        .tail(1)
        .sort_values("total_deaths_per_million", ascending=False)
    )

    display(latest[["location", "date", "total_deaths_per_million"]])

    fig, ax = plt.subplots(figsize=(11, 5.5))
    latest.set_index("location")["total_deaths_per_million"].plot(kind="bar", ax=ax)
    ax.set_title("Latest Available Confirmed COVID-19 Deaths per Million")
    ax.set_ylabel("Confirmed deaths per million")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=55)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    plt.show()


In [ ]:
# 10. Vaccination rollout
vax_col = COL["people_fully_vaccinated"]
pop_col = COL["population"]

if vax_col and pop_col:
    vax = work[[COL["location"], COL["date"], vax_col, pop_col]].copy()
    vax["full_protocol_pct"] = np.where(
        vax[pop_col] > 0,
        vax[vax_col] / vax[pop_col] * 100,
        np.nan
    )

    vax_latest = (
        vax.dropna(subset=["full_protocol_pct"])
        .sort_values("date")
        .groupby(COL["location"], as_index=False)
        .tail(1)
        .sort_values("full_protocol_pct", ascending=False)
    )

    display(vax_latest[[COL["location"], "date", "full_protocol_pct"]])

    fig, ax = plt.subplots(figsize=(11, 5.5))
    vax_latest.set_index(COL["location"])["full_protocol_pct"].plot(kind="bar", ax=ax)
    ax.set_title("Latest Available Share Completing Initial Vaccination Protocol")
    ax.set_ylabel("Population (%)")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=55)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    plt.show()


## 11. Wave detection

A practical portfolio approach is to define a wave using a smoothed series rather than raw daily reports:

- smooth daily new cases with a 7-day rolling mean;
- identify prominent local maxima;
- compare the timing and magnitude of peaks across countries;
- avoid interpreting reporting spikes as epidemiological changes without context.

The code below gives a simple peak-finding implementation. Thresholds can be adjusted for a more formal analysis.


In [ ]:
# 11.1 Simple wave / peak detection
def detect_peaks(country_df, value_col, min_gap_days=21):
    s = country_df[["date", value_col]].dropna().copy()
    if s.empty:
        return pd.DataFrame(columns=["date", value_col])

    s = s.sort_values("date").reset_index(drop=True)
    peaks = []
    for i in range(1, len(s)-1):
        if s.loc[i, value_col] > s.loc[i-1, value_col] and s.loc[i, value_col] >= s.loc[i+1, value_col]:
            peaks.append((s.loc[i, "date"], s.loc[i, value_col]))

    peaks = pd.DataFrame(peaks, columns=["date", value_col])
    if peaks.empty:
        return peaks

    # Keep the most prominent peak within each minimum-gap window.
    peaks = peaks.sort_values(value_col, ascending=False).reset_index(drop=True)
    chosen = []
    for _, row in peaks.iterrows():
        if all(abs((row["date"] - d).days) >= min_gap_days for d in chosen):
            chosen.append(row["date"])
    return peaks[peaks["date"].isin(chosen)].sort_values("date")

if "new_cases_7d" in work.columns:
    india_peaks = detect_peaks(work[work["location"] == "India"], "new_cases_7d")
    display(india_peaks.head(10))


## 12. Interpretation rules and assumptions

### Case fatality rate (CFR)
`confirmed deaths / confirmed cases × 100`

This is a **reported-case fatality ratio**, not an infection fatality rate. It is affected by testing coverage, reporting delays, age mix, treatment, and differences in case definitions.

### Doubling time
`ln(2) / growth rate`

It is only meaningful when cumulative cases are increasing. When growth is flat or negative, the notebook leaves doubling time as missing rather than creating a misleading number.

### Missing values
Missing observations are retained as missing. Interpolation is not used for the core source data because doing so can create artificial trends.

### Country comparison
Counts alone are not fair across countries of very different population sizes. The dashboard therefore emphasizes cases/deaths per million where appropriate.


In [ ]:
# 13. Export cleaned dataset
export_cols = [
    c for c in [
        "location", "date",
        COL["population"], COL["new_cases"], COL["total_cases"],
        COL["new_deaths"], COL["total_deaths"],
        COL["people_vaccinated"], COL["people_fully_vaccinated"],
        "new_cases_7d", "new_deaths_7d",
        "new_cases_per_million", "new_deaths_per_million",
        "total_cases_per_million", "total_deaths_per_million",
        "case_fatality_rate_pct", "doubling_time_days"
    ] if c and c in work.columns
]

cleaned_path = "../data/covid_cleaned_focus_countries.csv"
work[export_cols].to_csv(cleaned_path, index=False)

print("Saved:", cleaned_path)
print("Rows:", len(work))


## 14. Executive findings template

After running the notebook on the current OWID data, update the final findings with the exact values printed by the latest run.

Use plain-language findings such as:

1. **Wave timing:** identify the largest smoothed case peaks for the selected countries.
2. **Population normalization:** explain how per-million rates change the comparison versus raw totals.
3. **Vaccination rollout:** compare at-least-one-dose and completed-protocol coverage.
4. **Mortality:** compare confirmed deaths per million while clearly stating reporting limitations.
5. **Association, not causation:** describe whether high vaccination coverage and lower confirmed mortality appear together in the selected data without claiming that vaccination alone caused the difference.


## 15. Conclusion

This project demonstrates a complete real-world analytics workflow:

**source → cleaning → validation → derived metrics → trend analysis → visualization → business-friendly explanation**

The most important analytical discipline is to separate what the data directly show from what cannot be inferred causally. COVID-19 reporting varies across countries and over time, so normalized metrics, rolling averages, and transparent assumptions are essential.
